# Quick start

The shortest path from a video file to a tau cube with documented uncertainty. We use the synthetic plume that ships with the test suite so the notebook is reproducible without an external dataset.

In [ ]:
import numpy as np
import imageio.v2 as imageio
from pathlib import Path
import tempfile

# Build a 50-frame, 64x64, 16-bit TIFF stack with a known Gaussian
# plume (peak tau=0.5, sigma=5 px). Frames 0..19 are plume-free,
# 20..49 contain the plume.
yy, xx = np.mgrid[0:64, 0:64]
tau_gt = 0.5 * np.exp(-((xx - 32)**2 + (yy - 32)**2) / (2 * 5.0**2))
transmittance = np.exp(-tau_gt)

rng = np.random.default_rng(0)
frames = []
for i in range(50):
    clean = 5000.0 * (transmittance if i >= 20 else np.ones_like(transmittance))
    noisy = clean + rng.normal(0, np.sqrt(clean)) + rng.normal(0, 5.0, size=clean.shape)
    frames.append(np.clip(noisy, 0, 65535).astype(np.uint16))

tmpdir = Path(tempfile.mkdtemp())
video_path = tmpdir / 'plume.tif'
imageio.mimwrite(video_path, frames)
print(f'wrote {video_path}')

Run the full pipeline. The five public functions chain by return type.

In [ ]:
import smokesight as ss

cal    = ss.calibrate(video_path, {'sensor': {'gain': 0.012, 'bit_depth': 16, 'ner': 0.002}}, progress=False)
bg     = ss.background(cal, n_frames=20)
result = ss.retrieve(cal, bg, min_confidence=0.0)
print(result)

Visualise a single frame of tau and its uncertainty.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
im0 = axes[0].imshow(result.tau[30], cmap='RdBu_r', vmin=-0.1, vmax=0.6)
axes[0].set_title('tau, frame 30')
plt.colorbar(im0, ax=axes[0], label='Optical Depth')

im1 = axes[1].imshow(result.sigma_tau[30], cmap='viridis', vmin=0)
axes[1].set_title('sigma_tau, frame 30')
plt.colorbar(im1, ax=axes[1], label='1-sigma uncertainty')
plt.tight_layout()
plt.show()

Recovered peak tau at the plume centre, averaged over plume-present frames:

In [ ]:
peak = result.tau[20:, 32, 32]
peak = peak[~np.isnan(peak)]
print(f'recovered peak tau: {peak.mean():.3f} +/- {peak.std():.3f} (ground truth: 0.500)')

Write the result to a CF-1.9 NetCDF4 file. Dispersion modellers (HYSPLIT, FLEXPART, STILT) can consume this directly.

In [ ]:
out_nc = tmpdir / 'tau.nc'
result.to_netcdf(out_nc)

import xarray as xr
ds = xr.open_dataset(out_nc)
print(ds)
ds.close()